# 타이타닉 상관관계 시각화

여러 종류의 그래프로 **변수들이 서로 어떻게 연관되는지** 찾아봅니다.

- 수치형 vs 수치형: 산점도, 상관계수 히트맵, 페어플롯
- 범주형 vs 생존: 생존율 막대/히트맵
- 분포 비교: 바이올린, KDE
- 파생 변수: `FamilySize` (가족 수)

> 상관관계는 "함께 움직이는 정도"일 뿐, 인과관계는 아닙니다.

## 0. 라이브러리와 데이터 준비

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="AppleGothic")

DATA_PATH = "/Users/remchoi/.cache/kagglehub/datasets/heptapod/titanic/versions/1/train_and_test2.csv"
df = pd.read_csv(DATA_PATH).rename(columns={"2urvived": "Survived"})
zero_cols = [c for c in df.columns if c.startswith("zero")]
df = df.drop(columns=zero_cols + ["Passengerid"])
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["FamilySize"] = df["sibsp"] + df["Parch"] + 1
df.head()

## 1. 상관계수 히트맵 (Pearson)

- **빨강(+1)**: 한 변수가 커지면 다른 변수도 커짐
- **파랑(-1)**: 한 변수가 커지면 다른 변수는 작아짐
- **0 근처**: 직선 관계가 거의 없음

`annot=True` 로 숫자, `mask` 로 위쪽 삼각형을 가려 보기 쉽게 합니다.

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5)
plt.title("Pearson 상관계수 히트맵")
plt.tight_layout()
plt.show()

## 2. Pearson vs Spearman

- **Pearson**: 두 변수의 *직선* 관계
- **Spearman**: *순위* 관계 (단조 증가/감소면 값을 잡아냄)

`Fare` 처럼 한쪽으로 치우친 변수는 Spearman이 더 큰 값을 보여줄 수 있습니다.

In [ ]:
pearson = df.corr(numeric_only=True)["Survived"].drop("Survived")
spearman = df.corr(method="spearman", numeric_only=True)["Survived"].drop("Survived")

compare = pd.DataFrame({"Pearson": pearson, "Spearman": spearman})
compare = compare.reindex(compare.abs().max(axis=1).sort_values(ascending=False).index)

ax = compare.plot(kind="barh", figsize=(9, 6), color=["steelblue", "darkorange"])
ax.axvline(0, color="gray", linewidth=0.8)
ax.set_title("생존(Survived)과의 상관계수 비교")
ax.set_xlabel("상관계수")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

compare.round(3)

## 3. 페어플롯 (Pairplot)

여러 수치형 변수를 2차원 산점도로 한 번에 그려, 변수 짝마다 관계를 봅니다.
색(`hue`)은 생존 여부입니다.

In [ ]:
pair_cols = ["Age", "Fare", "Pclass", "FamilySize", "Survived"]
sns.pairplot(df[pair_cols], hue="Survived", palette="Set1", diag_kind="kde", corner=True)
plt.show()

## 4. 산점도: Fare vs Age

점 하나가 승객 한 명입니다. 색으로 생존 여부, 모양으로 성별을 구분합니다.
`alpha` 는 점을 반투명하게 해서 겹친 부분을 보이게 합니다.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="Age", y="Fare", hue="Survived", style="Sex",
                palette="Set1", alpha=0.6)
plt.title("Age vs Fare (색: 생존, 모양: 성별)")
plt.tight_layout()
plt.show()

## 5. 바이올린 플롯

상자그림 + 분포 모양을 함께 보여줍니다. `hue="Sex"` 로 성별까지 나눠 봅니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.violinplot(data=df, x="Survived", y="Age", hue="Sex", split=True,
               palette="Set2", ax=axes[0])
axes[0].set_title("생존 여부별 Age 분포 (성별)")
sns.violinplot(data=df, x="Survived", y="Fare", hue="Sex", split=True,
               palette="Set2", ax=axes[1])
axes[1].set_title("생존 여부별 Fare 분포 (성별)")
plt.tight_layout()
plt.show()

## 6. 범주형 조합 히트맵: Sex × Pclass 생존율

두 범주형 변수를 조합했을 때 생존율이 어떻게 달라지는지 봅니다.
`pivot_table` 로 교차표를 만들고 히트맵으로 그립니다.

In [ ]:
pivot = df.pivot_table("Survived", index="Sex", columns="Pclass", aggfunc="mean")

plt.figure(figsize=(7, 5))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1)
plt.title("Sex × Pclass 생존율")
plt.ylabel("Sex (0=남, 1=여)")
plt.xlabel("Pclass")
plt.tight_layout()
plt.show()

## 7. 생존율 막대: Embarked × Pclass

In [ ]:
def survival_rate(data, col):
    return data.groupby(col)["Survived"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
survival_rate(df, "Embarked").plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Embarked 별 생존율")
axes[0].set_ylabel("생존율")
axes[0].set_ylim(0, 1)

survival_rate(df, "Pclass").plot(kind="bar", ax=axes[1], color="salmon")
axes[1].set_title("Pclass 별 생존율")
axes[1].set_ylabel("생존율")
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 8. 파생 변수: FamilySize

`FamilySize = sibsp + Parch + 1` 로 가족 수를 만듭니다.
가족 수가 너무 적거나 많으면 생존율이 낮아지는 **비선형 관계**를 볼 수 있습니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

family_rate = df.groupby("FamilySize")["Survived"].mean()
family_rate.plot(kind="bar", ax=axes[0], color="mediumseagreen")
axes[0].set_title("FamilySize 별 생존율")
axes[0].set_ylabel("생존율")
axes[0].set_ylim(0, 1)

sns.countplot(data=df, x="FamilySize", hue="Survived", palette="Set2", ax=axes[1])
axes[1].set_title("FamilySize 별 인원 수")
plt.tight_layout()
plt.show()

df.groupby("FamilySize")["Survived"].agg(["mean", "count"]).round(3)

## 9. KDE: Pclass 별 Fare 분포

등급별 운임 분포가 어떻게 다른지 곡선으로 비교합니다.

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df, x="Fare", hue="Pclass", fill=True, common_norm=False, palette="viridis")
plt.title("Pclass 별 Fare 분포")
plt.xlim(-10, 200)
plt.tight_layout()
plt.show()

## 10. 상관관계 요약

| 관계 | 상관계수(생존) | 해석 |
|---|---|---|
| `Sex` | **+0.40** | 여성이 생존할 가능성이 훨씬 높음 (가장 강한 변수) |
| `Pclass` | -0.25 | 등급이 낮을수록(숫자가 클수록) 생존율 낮음 |
| `Fare` | +0.17 (Spearman +0.24) | 운임이 높을수록 생존율 높음 |
| `FamilySize` | +0.02 (Spearman +0.13) | 약하지만 비선형 (4명 부근에서 최고) |
| `Age` | -0.06 | 거의 관계 없음 (약한 음의 관계) |

### 핵심 발견
1. **Sex × Pclass 상호작용**: 여성이라도 3등급은 생존율 0.33으로, 1·2등급(0.63~0.66)보다 크게 낮음.
2. **FamilySize는 비선형**: 1명(0.21)이나 5명 이상은 낮고, 2~4명(0.37~0.49)에서 높음.
3. **Fare는 Pclass와 강한 음의 상관**: 비싼 표 = 좋은 등급 = 높은 생존율.
4. 단순 상관계수만으로는 `FamilySize` 같은 비선형 관계를 놓칠 수 있어 시각화가 중요합니다.